# Week 8 — Day 3: Quantisation

This notebook contains only the required Day 3 work.

## Required tasks
1. Create an 8-bit model
2. Create a 4-bit model
3. Create a GGUF model (`q4_0`)
4. Compare FP16, INT8, INT4, and GGUF on:
   - Size
   - Speed
   - Quality
5. Prepare `QUANTISATION-REPORT.md`

## Required deliverables
```text
quantized/model-int8/
quantized/model-int4/
quantized/model.gguf
QUANTISATION-REPORT.md
```

Use Google Colab with a T4 GPU.


## STEP 1 — Install required libraries


In [ ]:
!pip uninstall -y torchao
!pip install -q -U transformers peft accelerate bitsandbytes safetensors
print("Dependencies installed.")


## STEP 2 — Verify GPU


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU from Runtime -> Change runtime type.")

print("GPU:", torch.cuda.get_device_name(0))


## STEP 3 — Upload Day 2 adapter files


In [ ]:
from google.colab import files
import os, shutil

files.upload()

os.makedirs("/content/adapters", exist_ok=True)

for filename in ["adapter_model.safetensors", "adapter_config.json"]:
    src = f"/content/{filename}"
    dst = f"/content/adapters/{filename}"

    if os.path.exists(src):
        if os.path.exists(dst):
            os.remove(dst)
        shutil.move(src, dst)

!ls -lh /content/adapters


## STEP 4 — Reconstruct the fine-tuned model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH = "/content/adapters"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

print("Fine-tuned model reconstructed successfully.")


## STEP 5 — Merge LoRA into the base model


In [ ]:
merged_model = model.merge_and_unload()
print("LoRA merged successfully.")


## STEP 6 — Save temporary FP16 baseline


In [ ]:
import os

FP16_PATH = "/content/model-fp16-temp"
os.makedirs(FP16_PATH, exist_ok=True)

merged_model.save_pretrained(FP16_PATH, safe_serialization=True)
tokenizer.save_pretrained(FP16_PATH)

print("Temporary FP16 baseline saved.")


## STEP 7 — Measure FP16 size and speed


In [ ]:
import os, time

def folder_size_gb(path):
    total = 0
    for root, _, files in os.walk(path):
        for name in files:
            fp = os.path.join(root, name)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total / (1024**3)

def generate_text(model, prompt):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False
        )
    elapsed = time.time() - start

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    tokens_generated = len(generated)
    tps = tokens_generated / elapsed if elapsed > 0 else 0

    return response, elapsed, tps

TEST_PROMPT = "Explain Docker containers in simple terms."

fp16_response, fp16_latency, fp16_tps = generate_text(
    merged_model,
    TEST_PROMPT
)

fp16_size = folder_size_gb(FP16_PATH)

print("FP16 size (GB):", round(fp16_size, 3))
print("FP16 latency (s):", round(fp16_latency, 3))
print("FP16 tokens/sec:", round(fp16_tps, 3))
print("FP16 response:", fp16_response)


## STEP 8 — Create INT8 model


In [ ]:
from transformers import BitsAndBytesConfig

INT8_PATH = "/content/quantized/model-int8"
os.makedirs(INT8_PATH, exist_ok=True)

int8_config = BitsAndBytesConfig(load_in_8bit=True)

int8_model = AutoModelForCausalLM.from_pretrained(
    FP16_PATH,
    quantization_config=int8_config,
    device_map="auto"
)

int8_model.save_pretrained(INT8_PATH, safe_serialization=True)
tokenizer.save_pretrained(INT8_PATH)

print("INT8 model saved.")


## STEP 9 — Measure INT8 size and speed


In [ ]:
int8_response, int8_latency, int8_tps = generate_text(
    int8_model,
    TEST_PROMPT
)

int8_size = folder_size_gb(INT8_PATH)

print("INT8 size (GB):", round(int8_size, 3))
print("INT8 latency (s):", round(int8_latency, 3))
print("INT8 tokens/sec:", round(int8_tps, 3))
print("INT8 response:", int8_response)


## STEP 10 — Create INT4 model


In [ ]:
INT4_PATH = "/content/quantized/model-int4"
os.makedirs(INT4_PATH, exist_ok=True)

int4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

int4_model = AutoModelForCausalLM.from_pretrained(
    FP16_PATH,
    quantization_config=int4_config,
    device_map="auto"
)

int4_model.save_pretrained(INT4_PATH, safe_serialization=True)
tokenizer.save_pretrained(INT4_PATH)

print("INT4 model saved.")


## STEP 11 — Measure INT4 size and speed


In [ ]:
int4_response, int4_latency, int4_tps = generate_text(
    int4_model,
    TEST_PROMPT
)

int4_size = folder_size_gb(INT4_PATH)

print("INT4 size (GB):", round(int4_size, 3))
print("INT4 latency (s):", round(int4_latency, 3))
print("INT4 tokens/sec:", round(int4_tps, 3))
print("INT4 response:", int4_response)


## STEP 12 — Install llama.cpp


In [ ]:
!apt-get update -qq
!apt-get install -y -qq build-essential cmake git
%cd /content
!rm -rf llama.cpp
!git clone -q https://github.com/ggml-org/llama.cpp.git
!cmake -S llama.cpp -B llama.cpp/build -DCMAKE_BUILD_TYPE=Release
!cmake --build llama.cpp/build --config Release -j 2
print("llama.cpp built.")


## STEP 13 — Convert FP16 model to GGUF


In [ ]:
%cd /content/llama.cpp

!python convert_hf_to_gguf.py     /content/model-fp16-temp     --outfile /content/model-f16.gguf     --outtype f16

print("FP16 GGUF created.")


## STEP 14 — Quantize GGUF to Q4_0


In [ ]:
!mkdir -p /content/quantized

!./build/bin/llama-quantize     /content/model-f16.gguf     /content/quantized/model.gguf     Q4_0

print("GGUF Q4_0 model created.")
!ls -lh /content/quantized/model.gguf


## STEP 15 — Measure GGUF size and speed


In [ ]:
import subprocess, os, re, time

gguf_size = os.path.getsize("/content/quantized/model.gguf") / (1024**3)

cmd = [
    "/content/llama.cpp/build/bin/llama-cli",
    "-m", "/content/quantized/model.gguf",
    "-p", TEST_PROMPT,
    "-n", "100",
    "--temp", "0"
]

start = time.time()
result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)
gguf_latency = time.time() - start

gguf_output = result.stdout

print("GGUF size (GB):", round(gguf_size, 3))
print("GGUF latency (s):", round(gguf_latency, 3))
print("GGUF output:")
print(gguf_output)


## STEP 16 — Compare Size, Speed, and Quality


In [ ]:
print("===== SIZE COMPARISON =====")
print("FP16 :", round(fp16_size, 3), "GB")
print("INT8 :", round(int8_size, 3), "GB")
print("INT4 :", round(int4_size, 3), "GB")
print("GGUF :", round(gguf_size, 3), "GB")

print("\n===== SPEED COMPARISON =====")
print("FP16 tokens/sec :", round(fp16_tps, 3))
print("INT8 tokens/sec :", round(int8_tps, 3))
print("INT4 tokens/sec :", round(int4_tps, 3))
print("GGUF latency (s):", round(gguf_latency, 3))

print("\n===== QUALITY COMPARISON =====")
print("\nFP16:")
print(fp16_response)

print("\nINT8:")
print(int8_response)

print("\nINT4:")
print(int4_response)

print("\nGGUF:")
print(gguf_output)


## STEP 17 — Download required Day 3 deliverables


In [ ]:
%cd /content/quantized

!zip -qr model-int8.zip model-int8
!zip -qr model-int4.zip model-int4

from google.colab import files

files.download("/content/quantized/model-int8.zip")
files.download("/content/quantized/model-int4.zip")
files.download("/content/quantized/model.gguf")


# Final local structure

After extracting the two ZIP files locally:

```text
week8-llm-fine-tuning/
├── quantized/
│   ├── model-int8/
│   ├── model-int4/
│   └── model.gguf
└── QUANTISATION-REPORT.md
```

The temporary FP16 baseline is only used inside Colab for comparison and is not a required deliverable.
